# Encoding analysis: wilbur20210512

**Contents**

1. [Setup](#Setup)
2. [Single-variable models](#Single-variable-models)
   - Null - History - Trial type - Choice - Speed - Position - Trial progress
3. [Combined models](#Combined-models)
   - Full (using linear position) - Temporal (using trial progress)
4. [Model comparison](#Model-comparison)
   - Single-variable AIC/LRT - Drop-one & LRT


In [53]:
import os
import datajoint as dj
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir("..")
    
%matplotlib qt
%config InlineBackend.figure_format = 'retina'
sns.set_context("talk")

%reload_ext autoreload
%autoreload 2
pd.options.display.max_rows = 600
pd.set_option('display.float_format', lambda x: '%.9f' % x)

dj.config['display.limit'] = 10**3  

In [54]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats         
from scipy.stats import chi2 
from functools import partial
from notebooks.encoding_utils import *
from notebooks.spike_visualisation import *

## Setup

### Load data

In [55]:
base_dir = CONFIG["base_dir"]
BIN_SIZE = CONFIG["bin_size"]

data = load_and_prepare_data()
cov_df_correct = data["cov_df_correct"] #covariate dataframe, data from all correct trials
spike_counts_correct = data["spike_counts_correct"]
cov_df_common        = data["cov_df_common"] #covariate dataframe, conv_df_correct + runs_only mask
cov_df_out_common    = data["cov_df_out_common"] # only outbound trials, for choice analysis
spike_counts_common  = data["spike_counts_common"]
spike_counts_out_common = data["spike_counts_out_common"]
unit_ids             = data["unit_ids"]
bin_centers          = data["bin_centers"]
sp                   = data["scaling_params"]
pos_min_val, pos_max_val   = sp["pos_min"], sp["pos_max"]
speed_min_val, speed_max_val = sp["speed_min"], sp["speed_max"]

# trialized position : used by W-track visualization
trialized_position = pd.read_csv(
    f"{base_dir}/analysis/position/trialized_position.csv", index_col="time"
) # dataframe with position + behavior data from all four epochs, exported from multiple spyglass tables

print(f"cov_df_common: {len(cov_df_common):,} bins")
print(f"cov_df_out_common: {len(cov_df_out_common):,} bins (outbound)")

# model registry 
registry = build_model_registry()

cov_df_common: 430,792 bins
cov_df_out_common: 195,510 bins (outbound)


### Models:

#### Single-variable models:
1. Null model (constant rate)
2. Null model (outbound only, for comparison with other outbound-only models)
3. spike_count ~ history terms 
2. spike_count ~ trial_type (categorical)
3. spike_count ~ left/right choice (categorical)
4. spike_count ~ bs(speed, df = 4) (spline)
5. spike_count ~ bs(linear_position, df = 8) (spline)
6. spike_count ~ cr(progress, df= 6) (spline)

#### Mutli-variable models:
1. spike_count ~ trial_type + bs(speed) + bs(linear_position) 
2. spike_count ~ trial_type + bs(speed) + cr(trial_progress)

In [56]:
UNIT_LIST = [12, 47, 103, 211, 58] #for unit grid plots

## Data visualisation

In [11]:
plot_raster(spike_counts_correct, cov_df_correct, epoch = 2, units = UNIT_LIST, plot_position=True); #masked to include outboud/inbound trials, no zone or speed masking

In [57]:
plot_isi_grid(spike_counts_common, units = [115, 45, 118, 119, 22, 37], n_hist_bins=25, max_isi_ms=50);

In [58]:
plot_acg_grid(spike_counts_common, units=[115, 45, 118, 119, 22, 37], max_lag_ms=50, plot_bin_ms=2, epoch=2, cov_df=cov_df_common);

## Single-variable models

### Null model

#### Fit on one unit 

In [78]:
unit_idx = 18

results_constant = fit_single_unit(registry["null"]["formula"],
                                       cov_df_out_common, spike_counts_out_common, unit_idx)
print(results_constant.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               195510
Model:                            GLM   Df Residuals:                   195509
Model Family:                 Poisson   Df Model:                            0
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -72552.
Date:                Tue, 31 Mar 2026   Deviance:                       98796.
Time:                        16:24:29   Pearson chi2:                 1.72e+05
No. Iterations:                     6   Pseudo R-squ. (CS):         -2.220e-16
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.1334      0.007   -324.634      0.0

In [79]:
# Interpret the coefficient
mean_count_per_bin = np.exp(results_constant.params["Intercept"])
mean_rate_hz = mean_count_per_bin / BIN_SIZE

print(f"β₀ = {results_constant.params['Intercept']:.4f}")
print(f"exp(β₀) = {mean_count_per_bin:.4f} spikes/bin")
print(f"Firing rate = {mean_rate_hz:.2f} Hz")
print(f"Observed mean = {spk_cov_df_common['spike_count'].mean():.4f} spikes/bin")

β₀ = -2.1334
exp(β₀) = 0.1184 spikes/bin
Firing rate = 59.21 Hz
Observed mean = 0.0038 spikes/bin


In [80]:
# Diagnostics - residuals and KS test
plot_diagnostics(results = results_constant,
                 spike_counts=spike_counts_common[unit_idx],
                 cov_df = cov_df_common,
                 unit_label = str(unit_idx),
                 show_cumulative = True,)

In [81]:
formula = registry["null"]["formula"]

plot_diagnostics_batch(
    unit_list           = UNIT_LIST,  
    formula             = formula,
    cov_df              = cov_df_common,
    spike_counts_masked = spike_counts_common,
    unit_ids            = unit_ids,
)

#### Fit on all units

In [63]:
null_model_all = fit_glm_all_units("spike_count ~ 1",
                                   cov_df_common,
                                   spike_counts_common,
                                   unit_ids, model_name= "null",
                                   refit = False,
                                   fit_history=True) #whether to refit if already found on disk

Loading cached fit from /media/labuser/NA_1_2025/spyglass/wilbur/analysis/null_history_model_all.csv


In [64]:
COVARIATE_COLS     = ["linear_position", "speed"]
CATEGORICAL_COLS   = ["trial_type"]

In [65]:
diag_full_null, prof_null = load_model_outputs("null_model_all", fit_history=True)

In [67]:
plot_residual_profiles(prof_null, covariate_cols=COVARIATE_COLS) #average profiles across all cells


In [ ]:
panels = {
    "linear_position": "continuous",
    "speed":           "continuous",
    "trial_type":      "categorical",
}

plot_residual_heterogeneity(prof_null, panels=panels, row_normalise=True)


### History model

In [169]:
add_spike_history_mod = partial(add_spike_history, windows_ms=((0, 2), (2, 10), (10, 20), (20, 50))) #modified add_spike_history

In [70]:
history_model_formula = registry["null"]["formula"] + '+ hist_0_2ms + hist_2_10ms + hist_10_20ms + hist_20_50ms'
print(history_model_formula)

spike_count ~ 1+ hist_0_2ms + hist_2_10ms + hist_10_20ms + hist_20_50ms


In [71]:
unit_idx =115
spk_cov_df_common = cov_df_common.copy()
spk_cov_df_common["spike_count"] = spike_counts_common[unit_idx]

results_history = fit_single_unit(history_model_formula,
                                   cov_df_common,
                                   spike_counts_common, 
                                   unit_idx,
                                   per_unit_transform=add_spike_history_mod)
print(results_history.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               428026
Model:                            GLM   Df Residuals:                   428021
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -9897.9
Date:                Tue, 31 Mar 2026   Deviance:                       16548.
Time:                        16:22:02   Pearson chi2:                 3.98e+05
No. Iterations:                     9   Pseudo R-squ. (CS):           0.003632
Covariance Type:            nonrobust                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -5.7870      0.027   -213.335   

In [72]:
#interpret coefficients (exp(intercept + beta_)/bin_size = rate in Hz)
print("Baseline rate = ", np.exp(results_history.params["Intercept"])/0.002)
for name, coef in results_history.params[1:].items():
    print(f"Firing rate at {name} = ", np.exp(results_history.params["Intercept"] + coef)/0.002, f" | coefficient = {coef}")

Baseline rate =  1.5336546771839354
Firing rate at hist_0_2ms =  0.28308346938533413  | coefficient = -1.6896670449483309
Firing rate at hist_2_10ms =  5.945718151655176  | coefficient = 1.3550177571700048
Firing rate at hist_10_20ms =  4.4105295202805825  | coefficient = 1.0563411899303365
Firing rate at hist_20_50ms =  2.1721405992415415  | coefficient = 0.3480595676461752


In [73]:
# Diagnostics : history model
plot_diagnostics(
    results        = results_history,
    spike_counts   = spike_counts_common[unit_idx],
    cov_df         = cov_df_common,
)

In [ ]:
# problem: bursting cell simulations "explode"
windows = ((0,2),(2,10),(10,20),(20,50))
plot_predicted_isi(results_history, windows, title=f"History model: unit {unit_idx}", n_sim=50)


### Null (outbound only)

#### Fit on one unit

In [82]:
unit_idx = 9

results_constant_out = fit_single_unit(registry["null_outbound"]["formula"],
                                       cov_df_out_common, spike_counts_out_common, unit_idx)
print(results_constant_out.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               195510
Model:                            GLM   Df Residuals:                   195509
Model Family:                 Poisson   Df Model:                            0
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3073.7
Date:                Tue, 31 Mar 2026   Deviance:                       5283.3
Time:                        16:25:42   Pearson chi2:                 1.95e+05
No. Iterations:                     8   Pseudo R-squ. (CS):              0.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -6.1149      0.048   -127.097      0.0

In [83]:
# Interpret the coefficient
mean_count_per_bin = np.exp(results_constant_out.params["Intercept"])
mean_rate_hz = mean_count_per_bin / BIN_SIZE

print(f"β₀ = {results_constant_out.params['Intercept']:.4f}")
print(f"exp(β₀) = {mean_count_per_bin:.4f} spikes/bin")
print(f"Firing rate = {mean_rate_hz:.2f} Hz")

_out_df = cov_df_out_common.copy()
_out_df["spike_count"] = spike_counts_out_common[unit_idx]
print(f'Observed mean = {_out_df["spike_count"].mean():.4f} spikes/bin')

β₀ = -6.1149
exp(β₀) = 0.0022 spikes/bin
Firing rate = 1.10 Hz
Observed mean = 0.0022 spikes/bin


In [84]:
# Diagnostics — null outbound
plot_diagnostics(
    results        = results_constant_out,
    spike_counts   = spike_counts_out_common[unit_idx],
    cov_df         = cov_df_out_common,
)

In [85]:
formula = registry["null_outbound"]["formula"]

plot_diagnostics_batch(
    unit_list           = UNIT_LIST,
    formula             = formula,
    cov_df              = cov_df_out_common,
    spike_counts_masked = spike_counts_out_common,
    unit_ids            = unit_ids,
)

#### Fit on all units

In [86]:
null_model_out_all = fit_glm_all_units("spike_count ~ 1", cov_df_out_common, spike_counts_out_common, unit_ids, model_name="null_out", refit = False)

Loading cached fit from /media/labuser/NA_1_2025/spyglass/wilbur/analysis/null_out_model_all.csv


In [87]:
null_model_out_all.head(1)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,43029.533433157,-21513.766716578,34015.851199324,1.000000000,195510.000000000,True,{'Intercept': -3.769314375497827},{'Intercept': 0.01489058393604296},34015.851199324,0.000000000,NaN,null_out


In [ ]:
COVARIATE_COLS     = ["linear_position", "speed"]
CATEGORICAL_COLS   = ["trial_type"]

diag_null_out, prof_null_out = load_model_outputs("null_model_out_all")

In [90]:
plot_residual_profiles(prof_null_out, covariate_cols=COVARIATE_COLS)

In [ ]:
plot_residual_heterogeneity(prof_null_out, panels = panels)

### Trial type

#### Fit on one unit

In [96]:
unit_idx = 9

results_trial_type = fit_single_unit(registry["trial_type"]["formula"],
                                     cov_df_common, spike_counts_common, unit_idx)
print(results_trial_type.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               430792
Model:                            GLM   Df Residuals:                   430790
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10713.
Date:                Tue, 31 Mar 2026   Deviance:                       18122.
Time:                        16:27:49   Pearson chi2:                 4.29e+05
No. Iterations:                     8   Pseudo R-squ. (CS):          0.0006030
Covariance Type:            nonrobust                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -5

In [97]:
#interpret coefficients
rate_inbound  = np.exp(results_trial_type.params["Intercept"]) / BIN_SIZE        # Hz
rate_outbound = np.exp(results_trial_type.params["Intercept"] + results_trial_type.params["trial_type[T.outbound]"]) / BIN_SIZE
ratio         = np.exp(results_trial_type.params["trial_type[T.outbound]"])      # outbound/inbound rate ratio

print("inbound rate: ", rate_inbound)
print("outbound rate: ", rate_outbound)
print("outbound/inbound: ", ratio)

inbound rate:  2.5926335206371998
outbound rate:  1.1048028233904446
outbound/inbound:  0.4261315047407525


In [98]:
plot_categorical_comparison(results_trial_type, "trial_type", spk_cov_df_common)


In [99]:
# Diagnostics — trial_type model
plot_diagnostics(
    results        = results_trial_type,
    spike_counts   = spike_counts_common[unit_idx],
    cov_df         = cov_df_common,
)

/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1651: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(5, 5))


In [100]:
formula = registry["trial_type"]["formula"]

plot_diagnostics_batch(
    unit_list           = UNIT_LIST,
    formula             = formula,
    cov_df              = cov_df_common,
    spike_counts_masked = spike_counts_common,
    unit_ids            = unit_ids,
)

/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1500: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(


#### Fit all units 

In [101]:
trial_type_model_all = fit_glm_all_units("spike_count ~ trial_type", cov_df_common, spike_counts_common, unit_ids, model_name = "trial_type", refit = False)

Loading cached fit from /media/labuser/NA_1_2025/spyglass/wilbur/analysis/trial_type_model_all.csv


In [102]:
trial_type_model_all.head(1)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,100737.887388474,-50366.943694237,79288.999570058,2.000000000,430792.000000000,True,"{'Intercept': -3.6318901188331854, 'trial_type...","{'Intercept': 0.012672449437512294, 'trial_typ...",79338.721765096,1.000000000,NaN,trial_type


In [ ]:
diag_trial_type, prof_trial_type = load_model_outputs("trial_type_model_all")

/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1885: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="ks_D", order=model_order,
/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1894: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="drift_auc", order=model_order,
/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1901: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="ks_z_autocorr", order=model_order,

In [107]:
plot_residual_profiles(prof_trial_type, covariate_cols=CATEGORICAL_COLS)

In [ ]:
plot_residual_heterogeneity(prof_trial_type, panels=panels)

### Choice

#### Fit on one unit

In [113]:
unit_idx = 9

results_choice = fit_single_unit(registry["choice"]["formula"],
                                       cov_df_out_common, spike_counts_out_common, unit_idx)
print(results_choice.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               195510
Model:                            GLM   Df Residuals:                   195508
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3051.0
Date:                Tue, 31 Mar 2026   Deviance:                       5238.0
Time:                        16:29:55   Pearson chi2:                 1.95e+05
No. Iterations:                     8   Pseudo R-squ. (CS):          0.0002317
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -6.4876      0.081    -

In [114]:
#interpret coefficients
rate_left  = np.exp(results_choice.params["Intercept"]) / BIN_SIZE        # Hz
rate_right = np.exp(results_choice.params["Intercept"] + results_choice.params["choice[T.right]"]) / BIN_SIZE
ratio         = np.exp(results_choice.params["choice[T.right]"])      # outbound/inbound rate ratio

print("left rate: ", rate_left)
print("right rate: ", rate_right)
print("right/left: ", ratio)

left rate:  0.7610726181905239
right rate:  1.4734409616635504
right/left:  1.9360057456366078


In [116]:
plot_categorical_comparison(results_choice, "choice", spk_cov_df_common)


In [117]:
# Diagnostics — choice model
plot_diagnostics(
    results        = results_choice,
    spike_counts   = spk_cov_df_common["spike_count"].values,
    cov_df         = spk_cov_df_common,
)

In [118]:
formula = registry["choice"]["formula"]

plot_diagnostics_batch(
    unit_list           = UNIT_LIST,
    formula             = formula,
    cov_df              = cov_df_out_common,
    spike_counts_masked = spike_counts_out_common,
    unit_ids            = unit_ids,
)

#### Fit for all units

In [119]:
choice_model_all = fit_glm_all_units("spike_count ~ choice", cov_df_out_common, spike_counts_out_common, unit_ids, model_name = "choice", refit = False)

Loading cached fit from /media/labuser/NA_1_2025/spyglass/wilbur/analysis/choice_model_all.csv


In [ ]:
diag_choice, prof_choice = load_model_outputs("choice_model_all")

/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1885: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="ks_D", order=model_order,
/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1894: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="drift_auc", order=model_order,
/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1901: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="ks_z_autocorr", order=model_order,

In [122]:
plot_residual_profiles(prof_choice, covariate_cols={'choice'})

In [ ]:
plot_residual_heterogeneity(prof_choice, panels = {"linear_position": "continuous", "speed":"continuous", "choice":"categorical"})

### Speed (spline)

#### Fit on one unit


In [125]:
unit_idx = 9

from patsy import bs, cr

results_speed_spline = fit_single_unit(registry["speed_spline"]["formula"],
                                       cov_df_common, spike_counts_common, unit_idx)
print(results_speed_spline.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               430792
Model:                            GLM   Df Residuals:                   430787
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10532.
Date:                Tue, 31 Mar 2026   Deviance:                       17761.
Time:                        16:33:05   Pearson chi2:                 4.30e+05
No. Iterations:                     8   Pseudo R-squ. (CS):           0.001441
Covariance Type:            nonrobust                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

In [126]:
#interpret coefficients
speeds_of_interest = [10, 20, 30, 40, 60]
speeds_scaled = (np.array(speeds_of_interest) - speed_min_val) / (speed_max_val - speed_min_val)
rates = results_speed_spline.predict(pd.DataFrame({"speed_scaled": speeds_scaled})) / BIN_SIZE

print("Spline model — predicted rates:")
for s, r in zip(speeds_of_interest, rates):
    print(f"  speed={s:2f} cm/s : {r:.2f} Hz")

speed_range = np.linspace(speed_min_val, speed_max_val, 500)
speed_range_scaled = (speed_range - speed_min_val) / (speed_max_val - speed_min_val)
pred_curve = results_speed_spline.predict(pd.DataFrame({"speed_scaled": speed_range_scaled})) / BIN_SIZE
peak_speed = speed_range[np.argmax(pred_curve)]
print(f"\nPeak firing at: {peak_speed:.1f} cm/s ({pred_curve.max():.2f} Hz)")
print(f"Rate ratio high/low speed: {pred_curve.max() / pred_curve.min():.2f}x")

Spline model — predicted rates:
  speed=10.000000 cm/s : 2.47 Hz
  speed=20.000000 cm/s : 1.35 Hz
  speed=30.000000 cm/s : 0.93 Hz
  speed=40.000000 cm/s : 0.80 Hz
  speed=60.000000 cm/s : 0.98 Hz

Peak firing at: 119.5 cm/s (5.84 Hz)
Rate ratio high/low speed: 7.39x


In [127]:
# Diagnostics : speed spline
plot_diagnostics(
    results        = results_speed_spline,
    spike_counts   = spike_counts_common[unit_idx],
    cov_df         = cov_df_common,
)

In [128]:
formula = registry["speed_spline"]["formula"]

plot_diagnostics_batch(
    unit_list           = UNIT_LIST,
    formula             = formula,
    cov_df              = cov_df_common,
    spike_counts_masked = spike_counts_common,
    unit_ids            = unit_ids,
)

#### Fit on all units

In [129]:
speed_spline_model_all = fit_glm_all_units(registry["speed_spline"]["formula"],
                                            cov_df_common, spike_counts_common, unit_ids,
                                            model_name = "speed_spline",
                                            refit = False)

Loading cached fit from /media/labuser/NA_1_2025/spyglass/wilbur/analysis/speed_spline_model_all.csv


In [ ]:
diag_speed, prof_speed = load_model_outputs("speed_spline_model_all")

/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1885: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="ks_D", order=model_order,
/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1894: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="drift_auc", order=model_order,
/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1901: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="ks_z_autocorr", order=model_order,

In [131]:
plot_residual_profiles(prof_speed, covariate_cols=COVARIATE_COLS)

In [ ]:
plot_residual_heterogeneity(prof_speed, panels=panels)

### Linear position (spline)

#### Fit on one unit

In [134]:
unit_idx = 9

results_pos_spline = fit_single_unit(registry["pos_spline"]["formula"],
                                     cov_df_common, spike_counts_common, unit_idx)
print(results_pos_spline.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               430792
Model:                            GLM   Df Residuals:                   430783
Model Family:                 Poisson   Df Model:                            8
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10459.
Date:                Tue, 31 Mar 2026   Deviance:                       17614.
Time:                        16:33:57   Pearson chi2:                 4.30e+05
No. Iterations:                     8   Pseudo R-squ. (CS):           0.001782
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

In [136]:
_pos_run_cmp = trialized_position[trialized_position["zone"] == "run"][
    ["linear_position", "projected_x_position", "projected_y_position"]
].dropna().iloc[::5].copy()

plot_tuning_comparison(
    results_pos_spline,
    pd.DataFrame({"pos_scaled": np.linspace(0, 1, 200)}),
    "pos_scaled",
    "linear_position",
    spk_cov_df_common,
    min_val=pos_min_val,
    max_val=pos_max_val,
    xlabel="Position (cm)",
)
# plot_wtrack_comparison(unit_idx, spk_cov_df_common,
#                        results=results_pos_spline, pos_run=_pos_run_cmp) #### will not work withoug spyglass config

In [137]:
pos_range = np.linspace(pos_min_val, pos_max_val, 500)
pos_range_scaled = (pos_range - pos_min_val) / (pos_max_val - pos_min_val)  # changed
pred_rate = results_pos_spline.predict(pd.DataFrame({"pos_scaled": pos_range_scaled})) / BIN_SIZE  # changed

peak_pos  = pos_range[np.argmax(pred_rate)]
peak_rate = pred_rate.max()
print(f"Peak firing position: {peak_pos:.1f} cm, rate: {peak_rate:.2f} Hz")

Peak firing position: 269.1 cm, rate: 5.56 Hz


In [138]:
# Diagnostics — position spline
plot_diagnostics(
    results        = results_pos_spline,
    spike_counts   = spike_counts_common[unit_idx],
    cov_df         = cov_df_common,
)

In [139]:
formula = registry["pos_spline"]["formula"]

plot_diagnostics_batch(
    unit_list           = UNIT_LIST,
    formula             = formula,
    cov_df              = cov_df_common,
    spike_counts_masked = spike_counts_common,
    unit_ids            = unit_ids,
)

#### Fit on all units

In [140]:
pos_spline_model_all = fit_glm_all_units("spike_count ~ bs(pos_scaled, df=8)",
                                          cov_df_common, spike_counts_common, unit_ids,
                                          model_name="pos_spline",
                                          refit = False)

Loading cached fit from /media/labuser/NA_1_2025/spyglass/wilbur/analysis/pos_spline_model_all.csv


In [141]:
pos_spline_model_all.head(1)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,100018.574871328,-50000.287435664,78555.687052912,9.000000000,430792.000000000,True,"{'Intercept': -3.1763134432068973, 'bs(pos_sca...","{'Intercept': 0.06747874928540872, 'bs(pos_sca...",79338.721765096,8.000000000,NaN,pos_spline


In [ ]:
diag_pos, prof_pos = load_model_outputs("pos_spline_model_all")

/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1885: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="ks_D", order=model_order,
/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1894: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="drift_auc", order=model_order,
/media/labuser/NA_1_2025/spyglass/wilbur/notebooks/encoding_utils.py:1901: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=long_df, x="model", y="ks_z_autocorr", order=model_order,

In [143]:
plot_residual_profiles(prof_pos, covariate_cols=COVARIATE_COLS)

In [ ]:
plot_residual_heterogeneity(prof_pos, panels = panels)

### Trial progress (spline)

#### Fit on one unit

In [152]:
unit_idx = 9

results_tp_spline = fit_single_unit(registry["trial_progress_spline"]["formula"],
                                    cov_df_common, spike_counts_common, unit_idx)
print(results_tp_spline.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               430792
Model:                            GLM   Df Residuals:                   430785
Model Family:                 Poisson   Df Model:                            6
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10294.
Date:                Tue, 31 Mar 2026   Deviance:                       17284.
Time:                        16:37:30   Pearson chi2:                 4.34e+05
No. Iterations:                     9   Pseudo R-squ. (CS):           0.002546
Covariance Type:            nonrobust                                         
                                                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------

In [153]:
plot_tuning_comparison(
    results_tp_spline,
    pd.DataFrame({"trial_progress": np.linspace(0, 1, 200)}),
    "trial_progress",
    "trial_progress",
    spk_cov_df_common,
    xlabel="Trial progress",
)


In [154]:
# Diagnostics — trial progress spline
plot_diagnostics(
    results        = results_tp_spline,
    spike_counts   = spike_counts_common[unit_idx],
    cov_df         = cov_df_common,
)

In [155]:
formula = registry["trial_progress_spline"]["formula"]

plot_diagnostics_batch(
    unit_list           = UNIT_LIST,
    formula             = formula,
    cov_df              = cov_df_common,
    spike_counts_masked = spike_counts_common,
    unit_ids            = unit_ids,
)

#### Fit on all units

In [156]:
trial_progress_spline_model_all = fit_glm_all_units(f"spike_count ~ cr(trial_progress, df=6, constraints='center')",
                                                  cov_df_common, spike_counts_common, unit_ids,
                                                  model_name="trial_progress",
                                                  refit = False)

Loading cached fit from /media/labuser/NA_1_2025/spyglass/wilbur/analysis/trial_progress_model_all.csv


In [157]:
trial_progress_spline_model_all.head(1)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,99380.563188789,-49683.281594395,77921.675370373,7.000000000,430792.000000000,True,"{'Intercept': -3.758817804947291, ""cr(trial_pr...","{'Intercept': 0.01030771117634762, ""cr(trial_p...",79338.721765096,6.000000000,NaN,trial_progress


In [158]:
diag_tp, prof_tp = load_model_outputs("trial_progress_spline_model_all")

In [159]:
plot_residual_profiles(prof_tp, covariate_cols=COVARIATE_COLS)

In [ ]:
plot_residual_heterogeneity(prof_tp)

## Combined models

### Full model (trial type, position , speed)

#### Fit on one unit

In [170]:
full_model_formula = registry["full_model"]["formula"] + ' + hist_0_2ms + hist_2_10ms + hist_10_20ms + hist_20_50ms'
print(full_model_formula)

spike_count ~ trial_type + bs(pos_scaled, df=8) + bs(speed_scaled, df=4) + hist_0_2ms + hist_2_10ms + hist_10_20ms + hist_20_50ms


In [171]:
unit_idx = 45
results_full_pos = fit_single_unit(full_model_formula,
                                   cov_df_common, 
                                   spike_counts_common, 
                                   unit_idx,
                                   per_unit_transform=add_spike_history_mod)
print(results_full_pos.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               428026
Model:                            GLM   Df Residuals:                   428008
Model Family:                 Poisson   Df Model:                           17
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -16473.
Date:                Tue, 31 Mar 2026   Deviance:                       27248.
Time:                        16:40:32   Pearson chi2:                 4.14e+05
No. Iterations:                     9   Pseudo R-squ. (CS):           0.003085
Covariance Type:            nonrobust                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

In [172]:
#plot history coefficients
hist_params = {k: v for k, v in results_full_pos.params.items() if k.startswith("hist_")}
hist_labels = [k.replace("hist_", "").replace("_", "–").replace("ms", " ms") for k in hist_params]

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.bar(range(len(hist_params)), hist_params.values(), color="steelblue", edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(range(len(hist_params)))
ax.set_xticklabels(hist_labels)
ax.set_xlabel("Spike history window")
ax.set_ylabel("Coefficient (log scale)")
ax.set_title(f"History term coefficients — unit {unit_idx}")
plt.tight_layout()
plt.show()

In [173]:
# Diagnostics : full model
plot_diagnostics(
    results        = results_full_pos,
    spike_counts   = spike_counts_common[unit_idx],
    cov_df         = cov_df_common,
    show_cumulative=False
)

In [176]:
windows = ((0,2),(2,10),(10,20),(20,50)) #many simulations explode
plot_predicted_isi(results_full_pos, windows, title=f"Full model: unit {unit_idx}", n_sim=50)


#### Fit on all units

In [177]:
full_model_formula = registry["full_model"]["formula"] + ' + hist_0_2ms + hist_2_10ms + hist_10_20ms + hist_20_50ms'
full_model_all = fit_glm_all_units(full_model_formula, cov_df_common, spike_counts_common, unit_ids, model_name="full_model", refit = False, fit_history=True)

Loading cached fit from /media/labuser/NA_1_2025/spyglass/wilbur/analysis/full_model_history_model_all.csv


In [178]:
full_model_all.head(1)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,96681.929970117,-48322.964985058,75419.655857339,18.000000000,428026.000000000,True,"{'Intercept': -2.748615366303212, 'trial_type[...","{'Intercept': 0.0810530490796394, 'trial_type[...",78605.687609937,17.000000000,NaN,full_model_history


In [179]:
diag_full, prof_full = load_model_outputs("full_model_all", fit_history=True)

In [180]:
plot_residual_profiles(prof_full)

In [ ]:
plot_residual_heterogeneity(prof_full, row_normalise=True, panels=panels)

In [184]:
fig, axes, shared = plot_residual_rms_pair_models(
    "full_model",
    "full_model",
    fit_history_a=True,
    fit_history_b=False,
)


In [ ]:
# Marginal place tuning on W-track ###### will not work withoug spyglass config
# )
# pos_run = trialized_position[trialized_position["zone"] == "run"][
#     ["linear_position", "projected_x_position", "projected_y_position"]
# ].dropna().iloc[::5].copy()

# plot_marginal_tuning(
#     results_full_pos, "pos_scaled", "pos_scaled", "linear_position",
#     spk_cov_df_common, unit_idx=unit_idx,
#     min_val=pos_min_val, max_val=pos_max_val,
#     wtrack=True, pos_run=pos_run, title="full model", 

### Temporal full model (trial type, progress, speed)

#### Fit on one unit

In [192]:
temporal_model_formula = registry["temporal_model"]["formula"]
unit_idx = 9
results_temporal = fit_single_unit(temporal_model_formula,
                                   cov_df_common, spike_counts_common, unit_idx)
print(results_temporal.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               430792
Model:                            GLM   Df Residuals:                   430780
Model Family:                 Poisson   Df Model:                           11
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10117.
Date:                Tue, 31 Mar 2026   Deviance:                       16929.
Time:                        16:52:28   Pearson chi2:                 4.61e+05
No. Iterations:                     9   Pseudo R-squ. (CS):           0.003367
Covariance Type:            nonrobust                                         
                                                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------

In [193]:
# Diagnostics — temporal model
plot_diagnostics(
    results        = results_temporal,
    spike_counts   = spike_counts_common[unit_idx],
    cov_df         = cov_df_common,)

In [194]:
plot_diagnostics_batch(
    unit_list           = UNIT_LIST,
    formula             = temporal_model_formula,
    cov_df              = cov_df_common,
    spike_counts_masked = spike_counts_common,
    unit_ids            = unit_ids,
)

#### Fit on all units

In [195]:
temporal_model_all = fit_glm_all_units(temporal_model_formula, cov_df_common, spike_counts_common, unit_ids, model_name="temporal_model_all", refit = False)

Loading cached fit from /media/labuser/NA_1_2025/spyglass/wilbur/analysis/temporal_model_all_model_all.csv


In [196]:
diag_temporal, prof_temporal = load_model_outputs("temporal_model_all")

In [197]:
plot_residual_profiles(prof_temporal, covariate_cols=COVARIATE_COLS)

In [ ]:
plot_residual_heterogeneity(prof_temporal, panels = {'trial_progress': 'continuous',
 'speed': 'continuous',
 'trial_type': 'categorical'})

## Model comparison

### Single-variable

In [200]:
# Load all model CSVs
model_files = {
    "null":                    (f"{base_dir}/analysis/null_model_all.csv",                    dict(keep_default_na=False, na_values=[""])),
    "null_out":                (f"{base_dir}/analysis/null_out_model_all.csv",                dict(keep_default_na=False, na_values=[""])),
    "trial_type":              (f"{base_dir}/analysis/trial_type_model_all.csv",              {}),
    "choice":                  (f"{base_dir}/analysis/choice_model_all.csv",                  {}),
    "speed_spline":            (f"{base_dir}/analysis/speed_spline_model_all.csv",            {}),
    "pos_spline":              (f"{base_dir}/analysis/pos_spline_model_all.csv",              {}),
    "trial_progress_spline":   (f"{base_dir}/analysis/trial_progress_spline_model_all.csv",  {}),
    "temporal_model_all":      (f"{base_dir}/analysis/temporal_model_all.csv",               {}),
    "full_model_all":          (f"{base_dir}/analysis/full_model_all.csv",                   {}),
    
}

models = {}
for name, (path, kwargs) in model_files.items():
    df = pd.read_csv(path, index_col=0, **kwargs)
    df["model"] = name
    models[name] = df.set_index("unit")

# Null baseline per model — must match the dataset the model was fit on
null_for = {
    "trial_type":            "null",
    "speed_spline":          "null",
    "pos_spline":            "null",
    "trial_progress_spline": "null",
    "choice":                "null_out",
    "temporal_model_all":    "null",
    "full_model_all":        "null",
}

rows = []
for model_name, null_name in null_for.items():
    m   = models[model_name]
    nul = models[null_name]

    for uid in m.index:
        row      = m.loc[uid]
        null_row = nul.loc[uid]

        lrt_stat = 2 * (row["llf"] - null_row["llf"])
        lrt_df   = int(row["df_model"]) if pd.notna(row["df_model"]) else 0
        lrt_pval = (1 - chi2.cdf(lrt_stat, lrt_df)) if lrt_df > 0 else np.nan

        rows.append(dict(
            unit        = uid,
            model       = model_name,
            aic         = row["aic"],
            llf         = row["llf"],
            n_params    = row["n_params"],
            n_obs       = row["n_obs"],
            converged   = row["converged"],
            delta_aic   = row["aic"] - null_row["aic"],
            lrt_stat    = lrt_stat,
            lrt_df      = lrt_df,
            lrt_pval    = lrt_pval,
            tuned       = bool(lrt_pval < 0.05) if not np.isnan(lrt_pval) else False,
        ))

comparison = pd.DataFrame(rows)

summary = comparison.groupby("model").agg(
    mean_dAIC  = ("delta_aic", "mean"),
    median_dAIC= ("delta_aic", "median"),
    n_tuned    = ("tuned",     "sum"),
    frac_tuned = ("tuned",     "mean"),
    n_converged= ("converged", "sum"),
).round(3)
print(summary)

                           mean_dAIC    median_dAIC  n_tuned  frac_tuned  \
model                                                                      
choice                 -82.007000000  -17.163000000      197 0.749000000   
full_model_all        -537.947000000 -354.903000000      259 0.985000000   
pos_spline            -240.869000000 -141.534000000      250 0.951000000   
speed_spline          -238.277000000 -119.521000000      252 0.958000000   
temporal_model_all    -532.155000000 -292.211000000      260 0.989000000   
trial_progress_spline -365.011000000 -170.226000000      251 0.954000000   
trial_type             -54.171000000  -14.389000000      197 0.749000000   

                       n_converged  
model                               
choice                         262  
full_model_all                 258  
pos_spline                     259  
speed_spline                   262  
temporal_model_all             261  
trial_progress_spline          261  
trial_type         

In [201]:
# Pivot to wide AIC table : enables direct pairwise model comparison
aic_wide = comparison.pivot(index="unit", columns="model", values="aic")

# Add null baselines 
aic_wide["null"]     = models["null"]["aic"]
aic_wide["null_out"] = models["null_out"]["aic"]

# Best single-variable model per unit (all-trials models only)
all_trials_models = ["trial_type",  "speed_spline", "pos_spline", "trial_progress_spline", "temporal_model_all", "full_model_all"]
aic_wide["best_model"] = aic_wide[all_trials_models].idxmin(axis=1)
print("Best model counts (all units):")
print(aic_wide["best_model"].value_counts())

Best model counts (all units):
best_model
full_model_all           146
temporal_model_all       108
pos_spline                 5
trial_progress_spline      2
speed_spline               1
Name: count, dtype: int64


/tmp/ipykernel_2973269/1296881454.py:10: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  aic_wide["best_model"] = aic_wide[all_trials_models].idxmin(axis=1)


In [203]:
#full model improvement over null
glm_full = pd.read_csv(f"{base_dir}/analysis/full_model_all.csv", index_col=0)
glm_full = glm_full[glm_full["converged"] == True].copy()

df_model  = glm_full["n_params"] - 1                              # degrees of freedom vs null
lrt_stat  = glm_full["deviance_null"] - glm_full["deviance"]      # 2*(llf_full - llf_null)
lrt_pval  = 1 - chi2.cdf(lrt_stat, df_model)                      # per-unit p-value

n_sig  = (lrt_pval < 0.05).sum()
_, pop_pval = wilcoxon(lrt_stat, zero_method="wilcox", alternative="greater")  # population: LRT > 0?

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

# Panel A: LRT statistic distribution
ax = axes[0]
ax.hist(lrt_stat, bins=40, color="steelblue", edgecolor="none", alpha=0.8)
ax.axvline(lrt_stat.median(), color="orange", lw=1.5,
           label=f"median = {lrt_stat.median():.1f}")
ax.set_xlabel("LRT statistic")
ax.set_ylabel("Number of units")
ax.set_title(f"LRT: full vs null  (n={len(lrt_stat)} units)\nWilcoxon p = {pop_pval:.2e}")
ax.legend(fontsize=9)
sns.despine(ax=ax)

# Panel B: per-unit p-value histogram
ax = axes[1]
ax.hist(lrt_pval, bins=40, color="steelblue", edgecolor="none", alpha=0.8)
ax.axvline(0.05, color="red", lw=1.5, ls="--", label="p = 0.05")
ax.set_xlabel("Per-unit LRT p-value")
ax.set_ylabel("Number of units")
ax.set_title(f"{n_sig}/{len(lrt_pval)} units significant (p < 0.05)")
ax.legend(fontsize=9)
sns.despine(ax=ax)


### Drop-one & LRT (combined models)

In [214]:
# without history terms
datasets = {
    "common":   (cov_df_common, spike_counts_common),
}

drop_one_specs = make_drop_one_specs(
    datasets=datasets,
    model_names=["full_model"],
)

In [215]:
#check_if_exists=True skips already-saved CSVs
for model_name in drop_one_specs:
    run_drop_one_suite(drop_one_specs, model_name, base_dir, unit_ids, check_if_exists=True, )


Running drop-one suite: full_model_all(3 fits)
    Skipping trial_type - file exists
    Skipping bs(pos_scaled, df=8) - file exists
    Skipping bs(speed_scaled, df=4) - file exists


In [216]:
# Compute LRT for all models and concatenate 
lrt_frames = [compute_drop_one_lrt(model_name, base_dir, drop_one_specs) for model_name in drop_one_specs]
drop_one_results = pd.concat(lrt_frames, ignore_index=True)

print(drop_one_results.shape)


  [full_model_all drop=trial_type] skipped 6 units (non-converged)
  [full_model_all drop=bs(pos_scaled, df=8)] skipped 5 units (non-converged)
  [full_model_all drop=bs(speed_scaled, df=4)] skipped 5 units (non-converged)
(773, 9)


In [218]:
# Summary table 
summary_drop = (
    drop_one_results
    .groupby(["model", "dropped_term"])
    .agg(
        n_units          = ("unit",        "count"),
        n_significant    = ("significant", "sum"),
        frac_significant = ("significant", "mean"),
        mean_delta_aic   = ("delta_aic",   "mean"),
        median_delta_aic = ("delta_aic",   "median"),
    )
    .round(3)
)
print(summary_drop.to_string())

# Short labels for plotting 
term_labels = {
    "trial_type": "trial_type",
    "choice":     "choice",
    "bs(pos_scaled, df = 8)":                             "position",
    "bs(speed_scaled, df = 4)":                           "speed",
    "cr(trial_progress, df = 6, constraints = 'center')": "trial_progress",
}
plot_df = drop_one_results.copy()
plot_df["term_label"] = plot_df["dropped_term"].map(term_labels).fillna(plot_df["dropped_term"])

summary_plot = (
    plot_df.groupby(["model", "term_label"])
    .agg(frac_significant=("significant", "mean"), mean_delta_aic=("delta_aic", "mean"))
    .round(3)
)

# Figure: fraction significant + mean delta AIC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
summary_plot["frac_significant"].unstack("term_label").plot.barh(ax=ax)
ax.axvline(0.05, color="red", lw=1, ls="--", label="α=0.05")
ax.set_xlabel("fraction of units (LRT p < 0.05)")
ax.set_title("Unique contribution: fraction significant")
ax.legend(title="dropped term", fontsize=8, loc="lower right")
sns.despine(ax=ax)

ax = axes[1]
summary_plot["mean_delta_aic"].unstack("term_label").plot.barh(ax=ax)
ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_xlabel("mean ΔAIC (full − reduced)\nnegative = full model better")
ax.set_title("Unique contribution: effect size (AIC)")
ax.legend(title="dropped term", fontsize=8, loc="lower right")
sns.despine(ax=ax)

plt.tight_layout()

# Figure: per-unit delta AIC distributions 
g = sns.FacetGrid(plot_df, col="model", col_wrap=2, height=4, sharey=False, sharex=False)
g.map_dataframe(
    sns.boxplot, x="delta_aic", y="term_label",
    color="steelblue", width=0.5, flierprops=dict(marker=".", ms=3)
)
g.map(plt.axvline, x=0, color="red", lw=1, ls="--")
g.set_axis_labels("ΔAIC (full − reduced)", "dropped term")
g.set_titles("{col_name}")
sns.despine()
plt.tight_layout()

                                       n_units  n_significant  frac_significant  mean_delta_aic  median_delta_aic
model          dropped_term                                                                                      
full_model_all bs(pos_scaled, df=8)        258            252       0.977000000  -254.448000000    -147.207000000
               bs(speed_scaled, df=4)      258            243       0.942000000  -248.914000000    -105.285000000
               trial_type                  257            187       0.728000000   -49.237000000     -13.582000000


### Statistical summary

In [219]:
# FDR correction : 
drop_one_results = apply_fdr_correction(drop_one_results)


In [221]:
# Pairwise comparison : run for each combined model
pw_full     = pairwise_term_comparison(drop_one_results, 'full_model_all')

print('Full model pairwise q-values:')
print(pw_full.round(3).to_string())


Full model pairwise q-values:
                        bs(pos_scaled, df=8)  bs(speed_scaled, df=4)  trial_type
bs(pos_scaled, df=8)                     NaN             0.003000000 0.000000000
bs(speed_scaled, df=4)           0.003000000                     NaN 0.000000000
trial_type                       0.000000000             0.000000000         NaN


In [222]:
plot_drop_one_summary(drop_one_results, pairwise_pvals=pw_full,
                      heatmap_model='full_model_all')
